In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats

pd.set_option('display.max_columns', None)

In [2]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("threat_analysis")
    .getOrCreate()
    )

In [3]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

## Validação da Ameaça

In [4]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
df_threat = spark.read.parquet(threat_dataset_path)

In [5]:
df_threat.show()

+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+------------------+--------------+-----------------------+-----------------------+-------------+---------------+----------+---------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+-------------+---------------+--------------------+------------------+----------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+--------------------------------+--------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+---------------------+------------------+----------------------+--------------------------+-----------------------+---------------------------+-----------------------------+--------------------------+---------

In [6]:
df_possession_agg = (
    df_threat
    .groupBy('gameId', 'possession_id')
    .agg(F.countDistinct(F.col('eventId')).alias('qtd_eventos')
    )
    .sort('qtd_eventos', ascending=False)
)

print(df_possession_agg.count())
df_possession_agg.show()

116065
+------+-------------+-----------+
|gameId|possession_id|qtd_eventos|
+------+-------------+-----------+
|  4721|          216|         76|
|  4654|           32|         62|
|  4700|          323|         61|
|  4699|          130|         61|
|  4764|           75|         59|
|  4764|           47|         55|
|  4739|          224|         54|
|  4629|           41|         52|
|  4550|          262|         51|
|  4757|          146|         51|
|  4783|           40|         50|
|  4800|          284|         50|
|  4738|          289|         49|
|  4497|          277|         48|
|  4631|          111|         45|
|  4689|          133|         45|
|  4801|           25|         45|
|  4536|           59|         45|
|  4570|          332|         44|
|  4748|           92|         44|
+------+-------------+-----------+
only showing top 20 rows


In [7]:
df_possession_agg_pd = df_possession_agg.toPandas()

fig = go.Figure()

#cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig.add_trace(
    go.Box(
        y=df_possession_agg_pd['qtd_eventos'],
        name='',
        marker_color='#1f77b4',
        text=df_possession_agg_pd['possession_id'],
        hovertemplate='%{text}<br>Quantidade de eventos: %{y}<extra></extra>',
        #legendgroup=freq,
        #showlegend=True
    )
)

fig.update_layout(
    title=f'Ciclos de posse das partidas',
    height=800,
    width=1000
)

fig.update_yaxes(title_text='Quantidade de eventos')
fig.show()

In [8]:
(
    df_threat
    .filter((F.col('gameId') == 4721) & (F.col('possession_id') == 212))
).show(148)

+------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+---------------+---------------+-----------------------+-----------------------+-------------+---------------+----------+---------+-------------+---------------+----------------+-----------------+---------------------+--------------+-------------+------------+-------------+---------------+--------------------+------------------+----------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+--------------------------------+--------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+---------------------+------------------+----------------------+--------------------------+-----------------------+---------------------------+-----------------------------+--------------------------+---------

In [9]:
# delta ameaça por evento defensivo pra ver 
# considerar ciclos de posse pra saber se o evento defensivo aumentou ou diminuiu ameaça

In [10]:
(
    df_threat
    .groupBy('eventTypeDescription')
    .agg(
        F.round(F.min(F.col('threat_score')), 3).alias('min_threat_score'),
        F.round(F.avg(F.col('threat_score')), 3).alias('avg_threat_score'),
        F.round(F.median(F.col('threat_score')), 3).alias('median_threat_score'),        
        F.round(F.max(F.col('threat_score')), 3).alias('max_threat_score'),
        )
    .sort('median_threat_score', ascending=False)
).show(truncate=False)

+--------------------+----------------+----------------+-------------------+----------------+
|eventTypeDescription|min_threat_score|avg_threat_score|median_threat_score|max_threat_score|
+--------------------+----------------+----------------+-------------------+----------------+
|Shot                |0.336           |0.664           |0.682              |0.86            |
|Cross               |0.354           |0.568           |0.564              |0.854           |
|Touch Carry         |0.195           |0.497           |0.492              |0.832           |
|Challenge           |0.193           |0.485           |0.477              |0.854           |
|Ball Carry          |0.198           |0.468           |0.459              |0.796           |
|Pass                |0.193           |0.402           |0.383              |0.842           |
|Rebound             |0.193           |0.409           |0.382              |0.841           |
|Clearance           |0.192           |0.327           |0.31

In [11]:
def build_aggregated_df(
    df,
    group_cols,
    agg_cols,
    agg_func,
    agg_prefix
):
    """
    Agrega um DataFrame utilizando a função de agregação desejada.
    """

    # expressões de agregação com o prefixo conforme a agregação realizada (ex: média -> avg, máximo -> max)
    agg_exprs = [
        F.round(agg_func(F.col(c)), 3).alias(f"{agg_prefix}_{c}")
        for c in agg_cols
    ]

    # DataFrame agregado
    df_agg = (
        df
        .groupBy(*group_cols)
        .agg(*agg_exprs)
    )

    # nomes das colunas agregadas
    agg_cols_result = [
        f"{agg_prefix}_{c}"
        for c in agg_cols
    ]

    return agg_cols_result, df_agg


def join_match_stats(df, df_match_stats):

    # adiciona estatísticas da partida
    return (
        df.join(
            df_match_stats,
            on=[
                "date",
                "homeTeamName",
                "opponentTeamName"
            ],
            how="left"
        )
    )


def build_side_df(df, is_home, avg_cols):

    # filtra mandante ou visitante
    side_filter = F.col('homeTeam') if is_home else ~F.col('homeTeam')

    # mapeamento das colunas conforme o lado
    cols = {
        "teamName": "homeTeamName" if is_home else "opponentTeamName",
        "win": (F.col("FTR") == ('H' if is_home else 'A')),
        "goals": "FTHG" if is_home else "FTAG",
        "shots": "HS" if is_home else "AS",
        "shots_target": "HST" if is_home else "AST",
        "avg_win_odds": "AvgH" if is_home else "AvgA",
    }

    # colunas fixas
    fixed_select = [
        'competitionName',
        'season',
        'gameId',
        "date",
        F.col(cols["teamName"]).alias("teamName"),
    ]

    # métricas agregadas
    avg_select = [F.col(c) for c in avg_cols]

    # estatísticas da partida
    result_select = [
        cols["win"].alias("win"),
        F.col(cols["goals"]).alias("goals"),
        F.col(cols["shots"]).alias("shots"),
        F.col(cols["shots_target"]).alias("shots_target"),
        F.col(cols["avg_win_odds"]).alias("avg_win_odds"),
    ]

    return (
        df
        .filter(side_filter)
        .select(*fixed_select, *avg_select, *result_select)
    )

In [12]:
def plot_correlation_heatmap(df_agg, corr_cols):
    """
    Constrói o df agregado (via build_aggregated_df), converte para pandas,
    calcula a matriz de correlação das colunas em `corr_cols` e plota um
    heatmap com Plotly.
    """

    df_agg_pd = df_agg.toPandas()

    corr = df_agg_pd[corr_cols].corr(method='spearman')

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale='RdBu', 
            zmin=-1, 
            zmax=1,
            text=corr.round(2).values,
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title="Matriz de Correlação",
        template="simple_white",
        width=1500,
        height=900,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

In [13]:
def plot_pvalue_heatmap(df_agg, corr_cols, method='spearman', alpha=0.05):
    """
    Calcula a matriz de p-valor (via scipy) pro mesmo conjunto de colunas
    usado no heatmap de correlação, e plota com Plotly. Célula com
    p-valor < alpha recebe um "*" ao lado do número pra marcar significância.
    Usa pares completos (dropna por par de colunas), igual ao .corr() do pandas.
    """

    df_agg_pd = df_agg.toPandas()

    corr_func = stats.spearmanr if method == 'spearman' else stats.pearsonr

    # matriz de p-valor, mesma ordem/eixos do heatmap de correlação
    pvals = pd.DataFrame(np.nan, index=corr_cols, columns=corr_cols)

    for i, col_i in enumerate(corr_cols):
        for j, col_j in enumerate(corr_cols):
            if j < i:
                continue
            if i == j:
                # correlação de uma coluna com ela mesma é trivial (r=1, p=0);
                # calcular via scipy aqui quebraria (colunas duplicadas no select)
                pvals.loc[col_i, col_j] = 0.0
                continue
            paired = df_agg_pd[[col_i, col_j]].dropna()
            pvalue = np.nan if len(paired) < 3 else corr_func(paired[col_i], paired[col_j])[1]
            pvals.loc[col_i, col_j] = pvalue
            pvals.loc[col_j, col_i] = pvalue

    # texto da célula com "*" pra marcar p-valor < alpha
    text = pvals.round(3).astype(str)
    text = text.where(pvals >= alpha, text + '*')

    fig = go.Figure(
        data=go.Heatmap(
            z=pvals.values,
            x=pvals.columns,
            y=pvals.columns,
            colorscale='Blues',
            reversescale=True,
            zmin=0,
            zmax=1,
            text=text.values,
            texttemplate="%{text}",
            colorbar=dict(title="p-valor"),
        )
    )

    fig.update_layout(
        title=f"Matriz de p-valor ({method}) — * indica p < {alpha}",
        template="simple_white",
        width=1500,
        height=900,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

#### 1.2. Preparação da base de odds

In [14]:
# base de odds da Premier League 2022-2023
match_stats_22_23_path = str(data_folder_path / "match_stats" / "PL_22_23.csv")

df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(match_stats_22_23_path , sep=',')

In [15]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23
    .select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
    )
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
    .sort('date')
)

df_pl_match_stats_22_23_filtrado_mapped.show()

+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|      date|        homeTeamName|    opponentTeamName|FTHG|FTAG|FTR| HS| AS|HST|AST| AvgH| AvgA| AvgD|
+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|2022-08-05|      Crystal Palace|             Arsenal|   0|   2|  A| 10| 10|  2|  2| 4.39| 1.88| 3.59|
|2022-08-06|              Fulham|           Liverpool|   2|   2|  D|  9| 11|  3|  4|10.99| 1.28| 6.05|
|2022-08-06|     AFC Bournemouth|         Aston Villa|   2|   0|  H|  7| 15|  3|  2|  3.8| 2.04|  3.5|
|2022-08-06|        Leeds United|Wolverhampton Wan...|   2|   1|  H| 12| 15|  4|  6| 2.34| 3.18| 3.34|
|2022-08-06|    Newcastle United|   Nottingham Forest|   2|   0|  H| 23|  5| 10|  0| 1.67| 5.57|  3.8|
|2022-08-06|   Tottenham Hotspur|         Southampton|   4|   1|  H| 18| 10|  8|  2| 1.36| 8.64| 5.27|
|2022-08-06|             Everton|             Chelsea|   0|   1|  A|  8| 

In [16]:
# ============================================================
# Validação de completude do join com as estatísticas de partida
# ============================================================
# Confere, no nível de jogo (não de evento), se toda partida presente no
# threat_dataset encontrou uma linha correspondente na base de odds/stats
# (join em date + homeTeamName + opponentTeamName).

df_games_threat = (
    df_threat
    .select('gameId', 'date', 'homeTeamName', 'opponentTeamName')
    .distinct()
)

df_games_join_check = join_match_stats(
    df_games_threat,
    df_pl_match_stats_22_23_filtrado_mapped
)

total_games = df_games_threat.count()
unmatched_games = df_games_join_check.filter(F.col('FTHG').isNull())
unmatched_count = unmatched_games.count()

print(f'Jogos no threat_dataset: {total_games}')
print(f'Jogos sem correspondência na base de estatísticas: {unmatched_count}')

unmatched_games.select('gameId', 'date', 'homeTeamName', 'opponentTeamName').show(50, truncate=False)

Jogos no threat_dataset: 361
Jogos sem correspondência na base de estatísticas: 0
+------+----+------------+----------------+
|gameId|date|homeTeamName|opponentTeamName|
+------+----+------------+----------------+
+------+----+------------+----------------+



### 2. Criação das bases agregadas e teste de correlação

### 2.1. Por Time-Partida

In [17]:
group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam"
]

# zonas do campo consideradas nas colunas geradas no target_engineering
zone_suffixes = ['', '_half', '_third_2', '_third_3']

agg_cols = []
for suffix in zone_suffixes:
    agg_cols += [
        f"threat_score{suffix}",
        f"progression_dist{suffix}_norm",
        f"total_players{suffix}_norm",
        f"atk_def_advantage{suffix}_norm"
    ]

avg_cols, df_agg = build_aggregated_df(
    df=df_threat,
    group_cols=group_cols,
    agg_cols=agg_cols,
    agg_func=F.mean,
    agg_prefix="avg"
)

df_agg = join_match_stats(
    df_agg,
    df_pl_match_stats_22_23_filtrado_mapped
)

df_team_match = (
    build_side_df(df_agg, True, avg_cols)
    .unionByName(
        build_side_df(df_agg, False, avg_cols)
    )
)

corr_cols = (
    df_team_match
    .drop(*(group_cols + ["teamName"]))
    .columns
)

df_team_match.show(5)

plot_correlation_heatmap(df_team_match, corr_cols)
plot_pvalue_heatmap(df_team_match, corr_cols)

+---------------+---------+------+----------+----------------+----------------+-------------------------+----------------------+--------------------------+---------------------+------------------------------+---------------------------+-------------------------------+------------------------+---------------------------------+------------------------------+----------------------------------+------------------------+---------------------------------+------------------------------+----------------------------------+-----+-----+-----+------------+------------+
|competitionName|   season|gameId|      date|        teamName|avg_threat_score|avg_progression_dist_norm|avg_total_players_norm|avg_atk_def_advantage_norm|avg_threat_score_half|avg_progression_dist_half_norm|avg_total_players_half_norm|avg_atk_def_advantage_half_norm|avg_threat_score_third_2|avg_progression_dist_third_2_norm|avg_total_players_third_2_norm|avg_atk_def_advantage_third_2_norm|avg_threat_score_third_3|avg_progression_dis

### 2.2. Por Time-Partida-Ciclo de Posse

In [18]:
possession_group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam",
    "possession_id"
]

max_cols, df_possession = build_aggregated_df(
    df=df_threat,
    group_cols=possession_group_cols,
    agg_cols=agg_cols,
    agg_func=F.max,
    agg_prefix="max"
)

team_group_cols = [
    "gameId",
    "competitionName",
    "season",
    "date",
    "homeTeamName",
    "opponentTeamName",
    "homeTeam"
]

avg_cols, df_agg = build_aggregated_df(
    df=df_possession,
    group_cols=team_group_cols,
    agg_cols=max_cols,
    agg_func=F.mean,
    agg_prefix="avg"
)

df_agg = join_match_stats(
    df_agg,
    df_pl_match_stats_22_23_filtrado_mapped
)

df_team_match = (
    build_side_df(df_agg, True, avg_cols)
    .unionByName(
        build_side_df(df_agg, False, avg_cols)
    )
)

corr_cols = (
    df_team_match
    .drop(*(group_cols + ["teamName"]))
    .columns
)

df_team_match.show(5)

plot_correlation_heatmap(df_team_match, corr_cols)
plot_pvalue_heatmap(df_team_match, corr_cols)

+---------------+---------+------+----------+----------------+--------------------+-----------------------------+--------------------------+------------------------------+-------------------------+----------------------------------+-------------------------------+-----------------------------------+----------------------------+-------------------------------------+----------------------------------+--------------------------------------+----------------------------+-------------------------------------+----------------------------------+--------------------------------------+-----+-----+-----+------------+------------+
|competitionName|   season|gameId|      date|        teamName|avg_max_threat_score|avg_max_progression_dist_norm|avg_max_total_players_norm|avg_max_atk_def_advantage_norm|avg_max_threat_score_half|avg_max_progression_dist_half_norm|avg_max_total_players_half_norm|avg_max_atk_def_advantage_half_norm|avg_max_threat_score_third_2|avg_max_progression_dist_third_2_norm|avg_ma